# 1 Generate Noisy Data and Fitting

This is an optional notebook that is used to generate noisy data for the manuscript. We note that we provide the output of this notebook directly as a download. 

This notebook is used to generate and fit all the data required for the paper. We will generate data for the following noise cases: 

1, 2, 3, 4, 5, 6, 7, where each case corresponds to a different noise level. 

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

from m3util.util.IO import download_and_unzip

from belearn.dataset.dataset import BE_Dataset


import numpy as np


## Loading data for SHO fitting


In [4]:
# Download the data file from Zenodo
url = 'https://zenodo.org/record/7774788/files/PZT_2080_raw_data.h5?download=1'

# Specify the filename and the path to save the file
filename = '/data_raw5.h5'
save_path = './Data'

# download the file
download_and_unzip(filename, url, save_path)

Using files already downloaded


In [5]:
data_path = save_path + '/' + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path)

# print the contents of the file
dataset.print_be_tree

/
├ Measurement_000
  ---------------
  ├ Channel_000
    -----------
    ├ Bin_FFT
    ├ Bin_Frequencies
    ├ Bin_Indices
    ├ Bin_Step
    ├ Bin_Wfm_Type
    ├ Excitation_Waveform
    ├ Noise_Floor
    ├ Noisy_Data_1
    ├ Noisy_Data_2
    ├ Noisy_Data_3
    ├ Noisy_Data_4
    ├ Noisy_Data_5
    ├ Noisy_Data_6
    ├ Noisy_Data_7
    ├ Noisy_Data_8
    ├ Position_Indices
    ├ Position_Values
    ├ Raw_Data
    ├ Spatially_Averaged_Plot_Group_000
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spatially_Averaged_Plot_Group_001
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spectroscopic_Indices
    ├ Spectroscopic_Values
    ├ UDVS
    ├ UDVS_Indices
├ Noisy_Data_1_SHO_Fit
  --------

## Generates Noisy Data

This function will generate noisy records and save them as an h5_main file in the USID format. This allows the data to be computed with the Pycroscopy SHO Fitter. 

In [4]:
# calculates the standard deviation and uses that for the noise
noise_STD = np.std(dataset.raw_SHO_data)

# prints the standard deviation
print(noise_STD)

0.0038833667


In [5]:
dataset.raw_SHO_data.shape

(3600, 63360)

In [6]:
dataset.generate_noisy_data_records(noise_levels = np.arange(1,9), 
                                    verbose=True, 
                                    noise_STD=noise_STD)

Noise standard deviation: 0.0038833667058497667
The STD of the data is: 0.0038833667058497667
Adding noise level 1
Adding noise level 2
Adding noise level 3
Adding noise level 4
Adding noise level 5
Adding noise level 6
Adding noise level 7
Adding noise level 8


## SHO fits on all the datasets

This will take some time, Each fit takes about 10 minutes to complete. 

In [6]:
out = [f"Noisy_Data_{i}" for i in np.arange(1,9)]
out.append("Raw_Data")

for data in out:
    print(f"Fitting {data}")
    dataset.SHO_Fitter(dataset = data, h5_sho_targ_grp = f"{data}_SHO_Fit", max_mem=1024*64, max_cores= 20)

Fitting Noisy_Data_1
Working on:
./Data//data_raw5.h5
['Y', 'X'] [60, 60]


SHO Fits will be written to:
./Data/data_raw5.h5


Could not add group - it might already exist.
Consider calling test() to check results before calling compute() which computes on the entire dataset and writes results to the HDF5 file
	This class (likely) supports interruption and resuming of computations!
	If you are operating in a python console, press Ctrl+C or Cmd+C to abort
	If you are in a Jupyter notebook, click on "Kernel">>"Interrupt"
	If you are operating on a cluster and your job gets killed, re-run the job to resume



KeyboardInterrupt: 

80.13s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


### Checks the results to make sure it was saved correctly

In [8]:
# print the contents of the file
dataset.print_be_tree

/
├ Measurement_000
  ---------------
  ├ Channel_000
    -----------
    ├ Bin_FFT
    ├ Bin_Frequencies
    ├ Bin_Indices
    ├ Bin_Step
    ├ Bin_Wfm_Type
    ├ Excitation_Waveform
    ├ Noise_Floor
    ├ Noisy_Data_1
    ├ Noisy_Data_2
    ├ Noisy_Data_3
    ├ Noisy_Data_4
    ├ Noisy_Data_5
    ├ Noisy_Data_6
    ├ Noisy_Data_7
    ├ Noisy_Data_8
    ├ Position_Indices
    ├ Position_Values
    ├ Raw_Data
    ├ Spatially_Averaged_Plot_Group_000
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spatially_Averaged_Plot_Group_001
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spectroscopic_Indices
    ├ Spectroscopic_Values
    ├ UDVS
    ├ UDVS_Indices
├ Noisy_Data_1_SHO_Fit
  --------

In [9]:
dataset.file

'./Data//data_raw6.h5'

In [10]:
from m3util.util.h5 import find_measurement

In [12]:
find_measurement(dataset.file, "Noisy_Data_1",group = "Measurement_000/Channel_000")

'Noisy_Data_1'

In [5]:
from m3util.util.h5 import find_groups_with_string

In [14]:
find_groups_with_string(dataset.file, "Noisy_Data_1")

['/Noisy_Data_1_SHO_Fit', '/Noisy_Data_1_SHO_Fit/Noisy_Data_1-SHO_Fit_000']

In [15]:
find_groups_with_string(dataset.file, "Noisy_Data_2")

['/Noisy_Data_2_SHO_Fit', '/Noisy_Data_2_SHO_Fit/Noisy_Data_2-SHO_Fit_000']

In [16]:
find_groups_with_string(dataset.file, "Noisy_Data_9")

[]

In [10]:
find_groups_with_string(dataset.file, "Noisy_Data_2") == []

False

In [8]:
out2 = [f"Noisy_Data_{i}" for i in np.arange(1,9)]
out2.append("Raw_Data")

for data2 in out2:
    print(data2)
    #if find_groups_with_string(dataset.file, data2) == []:
    print(f"Fitting {data2}")
    dataset.SHO_Fitter(dataset = data2, h5_sho_targ_grp = f"{data2}_SHO_Fit", max_mem=1024*64, max_cores= 20)
    # else:
    #     print(f"{data2} already fit. Skipping...")



Noisy_Data_1
Fitting Noisy_Data_1
Working on:
./Data//data_raw5.h5
['Y', 'X'] [60, 60]


SHO Fits will be written to:
./Data/data_raw5.h5


Could not add group - it might already exist.
Consider calling test() to check results before calling compute() which computes on the entire dataset and writes results to the HDF5 file

Note: SHO_Fit has already been performed with the same parameters before. These results will be returned by compute() by default. Set override to True to force fresh computation

[<HDF5 group "/Measurement_000/Channel_000/Noisy_Data_1-SHO_Fit_000" (5 members)>]
SHO fits for Noisy_Data_1 already exist. Skipping.
Noisy_Data_2
Fitting Noisy_Data_2
Working on:
./Data//data_raw5.h5
['Y', 'X'] [60, 60]


SHO Fits will be written to:
./Data/data_raw5.h5


Could not add group - it might already exist.
Consider calling test() to check results before calling compute() which computes on the entire dataset and writes results to the HDF5 file
SHO fits for Noisy_Data_2 already ex

KeyboardInterrupt: 

In [22]:
import pyUSID as usid
import h5py


In [23]:
with h5py.File(dataset.file, "r+") as h5_file:

    h5_main = usid.hdf_utils.find_dataset(h5_file, dataset)[0]


TypeError: dset_name should be a string